In [ ]:
import os, sys, time, json, gc
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch_geometric.data import Data

import config
from data_download import download_raw_data
from graph_builder import build_graph_files, load_graph_files
from model import build_model
from trainer import run_training, evaluate
from results import save_all_results, print_results
from imputation_eval import evaluate_imputation
from fairness import (check_demographic_parity_pre,
                      check_demographic_parity_post,
                      check_fairness_metrics_post,
                      plot_demographic_parity)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Dataset: {config.DATASET_NAME}')
print(f'Sources: {config.BATCH_SOURCES}')

# ═══════════════════════════════════════════════════════════════════
#  EXPERIMENT CONTROL
#  Comment out a line (add #) or set to False to skip that section.
# ═══════════════════════════════════════════════════════════════════
RUN_NULL_INJECTION    = True  # Section 2 — inject MCAR nulls (run once per seed set)
RUN_ALL_SOURCES       = True   # Section 3 — full batch pipeline (all BATCH_SOURCES × seeds)
RUN_LAYERS_EXPERIMENT = True   # Section 4 — sweep NUM_LAYERS, one full run per layer count
RUN_LAYERS_DEPTH_EXPERIMENT = True   # Section 5 — sweep hidden layer depth


NUM_LAYERS_TO_TEST = [3]   # hidden layers to test (e.g. 3 → 3 hidden layers)

print('\nExperiment control:')
print(f'  RUN_NULL_INJECTION          = {RUN_NULL_INJECTION}')
print(f'  RUN_ALL_SOURCES             = {RUN_ALL_SOURCES}')
print(f'  RUN_LAYERS_EXPERIMENT       = {RUN_LAYERS_EXPERIMENT}')
print(f'  RUN_LAYERS_DEPTH_EXPERIMENT = {RUN_LAYERS_DEPTH_EXPERIMENT}')



In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  OPTION B — No Balancing
# ───────────────────────────────────────────────────────────────────
#  Original sex ratio preserved proportionally in every partition.
#  No rows added or removed — pure splitting.
#  Output folder: <dataset>_no_balance/
# ═══════════════════════════════════════════════════════════════════

import config
from balance_no_balance import load_and_split

config.set_balance_strategy('no_balance')
_load_data = load_and_split
load_and_split()


In [3]:
# # ═══════════════════════════════════════════════════════════════════
# #  OPTION A — Two-Stage Male-Only Downsampling
# # ───────────────────────────────────────────────────────────────────
# #  Stage 1: downsample male label=1 so P(1|Male) ≈ P(1|Female).
# #  Result: DP gap = 0 before the GNN.
# #  Output folder: <dataset>_twostage/
# # ═══════════════════════════════════════════════════════════════════

# import config
# from balance_twostage import load_and_split as _ts_load

# config.set_balance_strategy('twostage')
# _load_data = _ts_load
# _ts_load()


In [4]:
# # ═══════════════════════════════════════════════════════════════════
# #  OPTION C — Stratified Sampling
# # ───────────────────────────────────────────────────────────────────
# #  Divide full dataset by sensitive attribute (sex).
# #  Identify minority stratum → keep all rows.
# #  Downsample every majority stratum to match minority size.
# #  Merge strata → equal group sizes, no feature values modified.
# #  Note: positive rates within groups are NOT equalized.
# #  Output folder: <dataset>_stratified_sampling/
# # ═══════════════════════════════════════════════════════════════════

# import config
# from balance_stratified_sampling import load_and_split as _ss_load

# config.set_balance_strategy('stratified_sampling')
# _load_data = _ss_load
# _ss_load()


In [5]:
# # ═══════════════════════════════════════════════════════════════════
# #  OPTION D — Random Feature Redistribution
# # ───────────────────────────────────────────────────────────────────
# #  Keep ALL rows — no removal.
# #  majority_size - (total_size / number_of_groups)
# #  Randomly select rows from majority group; reassign their
# #  sensitive attribute value to the minority group label.
# #  All other feature values unchanged.
# #  Result: group sizes approximately equal; positive rates shift
# #  because transferred rows carry their original labels.
# #  Output folder: <dataset>_random_feature_redistribution/
# # ═══════════════════════════════════════════════════════════════════

# import config
# from balance_random_feature_redistribution import load_and_split as _rfr_load

# config.set_balance_strategy('random_feature_redistribution')
# _load_data = _rfr_load
# _rfr_load()


## Null Injection — Create MCAR Datasets

Injects missing values (MCAR) into the training split at each rate in `config.MISSING_RATE`.
Run once; subsequent runs skip files that already exist.
**Skip:** set `RUN_NULL_INJECTION = False` in Cell 1.

In [ ]:
if not RUN_NULL_INJECTION:
    print('⏭  Skipping null injection  (RUN_NULL_INJECTION = False)')
else:
    from null_injector import run_null_injection

    needed_rates = sorted(
        config.MISSING_RATE
        if isinstance(config.MISSING_RATE, list)
        else [config.MISSING_RATE]
    )

    seeds = config.SEEDS
    sep = '═' * 60
    print(f'\n{sep}')
    print(f'  NULL INJECTION — ALL SEEDS')
    print(f'  Strategy : {config.BALANCE_STRATEGY}')
    print(f'  Seeds    : {seeds}')
    print(f'  Rates    : {needed_rates}%')
    print(f'  Total    : {len(seeds)} seeds × {len(needed_rates)} rates = '
          f'{len(seeds)*len(needed_rates)} files')
    print(f'{sep}\n')

    original_seed         = config.RANDOM_SEED
    original_missing_rate = config.MISSING_RATE
    last_seed_processed   = None

    for seed_i, seed in enumerate(seeds, 1):
        config.set_seed(seed)

        missing_rates = [r for r in needed_rates
                         if not os.path.exists(config.mcar_train_csv(r))]
        if not missing_rates:
            print(f'  ✓ seed={seed} — all {len(needed_rates)} rates exist — skipping')
            continue

        print(f'\n{"▓"*10}  SEED {seed_i}/{len(seeds)}  seed={seed}  {"▓"*10}')

        _paths = [config.TRAIN_CSV_FILE, config.VAL_CSV_FILE, config.TEST_CSV_FILE]
        balanced_csv = os.path.join(config.DATA_DIR, f'{config.DATASET_NAME}_balanced.csv')
        if os.path.exists(balanced_csv):
            _paths.append(balanced_csv)
        for path in _paths:
            if os.path.exists(path):
                os.remove(path)
        _load_data()

        for rate in needed_rates:
            folder = config.mcar_folder(rate)
            if os.path.isdir(folder):
                shutil.rmtree(folder)
                print(f'  Removed old folder: {folder}/')

        clean_train_df = pd.read_csv(config.TRAIN_CSV_FILE)
        print(f'  Injecting rates {needed_rates}% for seed={seed} ...')
        config.MISSING_RATE = needed_rates
        # ── Columns to inject nulls into ─────────────────────────────────────
        # List columns to EXCLUDE from injection. All other feature columns
        # (except the target) will receive nulls.
        # Example: EXCLUDE_FROM_INJECTION = {'age', 'income'}
        EXCLUDE_FROM_INJECTION = set()  # ← edit this set
        NULL_INJECT_COLS = (
            None if not EXCLUDE_FROM_INJECTION
            else [c for c in clean_train_df.columns
                  if c not in EXCLUDE_FROM_INJECTION | {config.TARGET_COL}]
        )
        if NULL_INJECT_COLS is not None:
            print(f'  Injecting into {len(NULL_INJECT_COLS)} cols '
                  f'(excluded: {EXCLUDE_FROM_INJECTION})')
        run_null_injection(clean_train_df, run_impute=False, inject_cols=NULL_INJECT_COLS)

        for rate in needed_rates:
            print(f'    Saved → {config.mcar_folder(rate)}/')
        last_seed_processed = seed

    config.set_seed(original_seed)
    config.MISSING_RATE = original_missing_rate

    if last_seed_processed is not None and last_seed_processed != original_seed:
        print(f'\n  Rebuilding splits for original seed={original_seed} ...')
        _paths = [config.TRAIN_CSV_FILE, config.VAL_CSV_FILE, config.TEST_CSV_FILE]
        balanced_csv = os.path.join(config.DATA_DIR, f'{config.DATASET_NAME}_balanced.csv')
        if os.path.exists(balanced_csv):
            _paths.append(balanced_csv)
        for path in _paths:
            if os.path.exists(path):
                os.remove(path)
        _load_data()

    print('\n✓ Null datasets ready')


## Run All Sources — Batch Pipeline

Runs the full pipeline (impute → graph → train → eval → fairness) for every
`(source, imputation)` pair in `config.BATCH_SOURCES`, repeated across all seeds.
**Skip:** set `RUN_ALL_SOURCES = False` in Cell 1.

In [ ]:
all_summaries = []
if not RUN_ALL_SOURCES:
    print('⏭  Skipping batch pipeline  (RUN_ALL_SOURCES = False)')
else:
    # ── Determine effective seeds and sources based on RUN_NULL_INJECTION ────
    if not RUN_NULL_INJECTION:
        # No null data exists → single seed, clean sources only
        _seeds          = [config.SEEDS[0] if config.SEEDS else config.RANDOM_SEED]
        _sources_to_run = [(s, imp) for s, imp in config.BATCH_SOURCES
                           if not s.startswith('mcar_')]
        print(f'[batch] RUN_NULL_INJECTION=False → '
              f'using seed={_seeds[0]}, skipping mcar sources')
        if not _sources_to_run:
            print('[batch] ✗ No non-mcar sources in BATCH_SOURCES — nothing to run')
            all_summaries = []
            _skip_batch = True
        else:
            _skip_batch = False
    else:
        _seeds          = list(config.SEEDS)
        _sources_to_run = list(config.BATCH_SOURCES)
        _skip_batch     = False

    if _skip_batch:
        all_summaries = []
    else:
        def cleanup(model, data, device, source):
            if model is not None:
                model.cpu(); del model
            if data is not None:
                data.cpu(); del data
            plt.close('all')
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        def _wipe_results_only():
            if not os.path.isdir(config.RESULTS_DIR):
                return
            keep_suffixes = ('_batch_summary.csv', '_batch_summary.json')
            for fname in os.listdir(config.RESULTS_DIR):
                if fname.endswith(keep_suffixes):
                    continue
                path = os.path.join(config.RESULTS_DIR, fname)
                if os.path.isfile(path):
                    os.remove(path)

        all_summaries = []
        t_total = time.time()

        sep70 = '═' * 70
        print(f'\n{sep70}')
        print(f'  MULTI-SEED BATCH — {config.DATASET_NAME}')
        print(f'  Seeds  : {_seeds}  ({len(_seeds)} total)')
        print(f'  Sources: {len(_sources_to_run)}  per seed')
        print(f'  Total  : {len(_seeds) * len(_sources_to_run)} runs')
        print(f'{sep70}')

        for seed_idx, seed in enumerate(_seeds, 1):
            print(f'\n\n{"▓"*70}')
            print(f'  SEED {seed_idx}/{len(_seeds)}  —  seed={seed}')
            print(f'{"▓"*70}')

            config.set_seed(seed)

            _paths = [config.TRAIN_CSV_FILE, config.VAL_CSV_FILE, config.TEST_CSV_FILE]
            balanced_csv = os.path.join(config.DATA_DIR, f'{config.DATASET_NAME}_balanced.csv')
            if os.path.exists(balanced_csv):
                _paths.append(balanced_csv)
            for path in _paths:
                if os.path.exists(path):
                    os.remove(path)
            _load_data()
            _wipe_results_only()

            for i, (source, imputation) in enumerate(_sources_to_run, 1):
                print(f'\n{"█"*70}')
                print(f'  SEED {seed}  —  RUN {i}/{len(_sources_to_run)}  —  '
                      f"SOURCE='{source}'  IMPUTATION='{imputation}'")
                print(f'{"█"*70}')

                model = None
                data  = None
                try:
                    t_start = time.time()
                    config.reconfigure(source, imputation)
                    config.set_global_seed()

                    if config.RESOLVED_IMPUTATION != 'none':
                        if not os.path.exists(config.SOURCE_CSV):
                            parts = config.SOURCE.split('_')
                            rate = int(parts[1])
                            raw_csv = config.mcar_csv(rate)
                            from imputer import impute
                            df_raw = pd.read_csv(raw_csv)
                            df_imp = impute(df_raw, strategy=config.RESOLVED_IMPUTATION)
                            os.makedirs(config.GRAPH_DIR, exist_ok=True)
                            df_imp.to_csv(config.SOURCE_CSV, index=False)
                            print(f'✓ Imputed CSV saved → {config.SOURCE_CSV}')
                        else:
                            print(f'✓ Imputed CSV exists: {config.SOURCE_CSV}')

                    evaluate_imputation()
                    build_graph_files(force=True)

                    X, y, edge_index, train_mask, val_mask, test_mask, meta = load_graph_files()
                    data = Data(
                        x          = torch.tensor(X, dtype=torch.float),
                        edge_index = edge_index,
                        y          = torch.tensor(y, dtype=torch.long),
                        train_mask = train_mask,
                        val_mask   = val_mask,
                        test_mask  = test_mask,
                    ).to(device)
                    del X, y, edge_index

                    model = build_model(
                        num_features=meta['num_features'],
                        num_classes=meta['num_classes'],
                    ).to(device)

                    if os.path.exists(config.MODEL_F):
                        os.remove(config.MODEL_F)
                    history = run_training(model, data, device)

                    eval_results = evaluate(model, data, device)
                    print_results(eval_results)
                    save_all_results(history, eval_results, meta)

                    dp_pre   = check_demographic_parity_pre(test_mask)
                    dp_post  = check_demographic_parity_post(eval_results)
                    fair_ext = check_fairness_metrics_post(eval_results)
                    plot_demographic_parity(dp_pre, dp_post)

                    elapsed = time.time() - t_start
                    summary = {
                        'seed'                  : seed,
                        'no_layers'             : config.NUM_LAYERS,
                        'source'                : source,
                        'imputation'            : config.RESOLVED_IMPUTATION,
                        'test_acc'              : round(eval_results['test']['acc'], 4),
                        'test_f1'               : round(eval_results['test']['f1_macro'], 4),
                        'test_auc'              : round(eval_results['test']['roc_auc'], 4)
                                                  if eval_results['test'].get('roc_auc') else None,
                        'dp_pre'                : dp_pre['dp_difference'],
                        'dp_post'               : dp_post['dp_difference'],
                        'dp_pre_group_rates'    : dp_pre['group_rates'],
                        'dp_post_group_rates'   : dp_post['group_rates'],
                        'tpr_parity_gap'        : fair_ext['tpr_parity_gap']         if fair_ext else None,
                        'fpr_parity_gap'        : fair_ext['fpr_parity_gap']         if fair_ext else None,
                        'equal_opportunity_diff': fair_ext['equal_opportunity_diff'] if fair_ext else None,
                        'equalized_odds_diff'   : fair_ext['equalized_odds_diff']    if fair_ext else None,
                        'tpr_by_group'          : fair_ext['tpr_by_group']           if fair_ext else {},
                        'fpr_by_group'          : fair_ext['fpr_by_group']           if fair_ext else {},
                        'elapsed_sec'           : round(elapsed, 1),
                    }
                    all_summaries.append(summary)
                    print(f"\n✓ seed={seed} '{source}' done in {elapsed:.1f}s  |  "
                          f"Acc={summary['test_acc']}  F1={summary['test_f1']}  "
                          f"DP={summary['dp_post']}  EqOdds={summary['equalized_odds_diff']}")

                except Exception as exc:
                    import traceback
                    print(f'\n✗ FAILED: seed={seed} {source} — {exc}')
                    traceback.print_exc()
                    all_summaries.append({'seed': seed, 'source': source,
                                          'imputation': imputation, 'error': str(exc)})
                finally:
                    cleanup(model, data, device, source)

        print(f'\n{sep70}')
        print(f'  ALL DONE — {len(_seeds)} seed(s) × {len(_sources_to_run)} sources '
              f'in {time.time()-t_total:.1f}s')
        print(f'{sep70}')



## Results Table — All Sources

Summary table + CSV/JSON for the batch pipeline results above.
Only runs when `all_summaries` is non-empty (i.e. Section 3 was executed).

In [ ]:
if not all_summaries:
    print('⏭  No batch results to display (all_summaries is empty)')
else:
    def _rate(group_rates, group_name):
        if group_rates and group_name in group_rates:
            return group_rates[group_name]
        return float('nan')

    sep = '═' * 140
    print(f'\n{sep}')
    print(
        f"  {'Seed':>6} {'Source':<20} {'Imputation':<12}"
        f" {'Test Acc':>9} {'Test F1':>8}"
        f" {'DP Pre':>8} {'DP Post':>8}"
        f" {'EqOdds':>7} {'EqOpp':>7}"
        f" {'TPR-M':>6} {'TPR-F':>6}"
        f" {'Time':>7}"
    )
    print(f"  {'─' * 132}")

    for s in all_summaries:
        if 'error' in s:
            print(f"  {str(s.get('seed','?')):>6} {s['source']:<20} FAILED: {s['error']}")
            continue
        tpr_by = s.get('tpr_by_group', {}) or {}
        print(
            f"  {str(s['seed']):>6} {s['source']:<20} {s['imputation']:<12}"
            f" {s['test_acc']:>9.4f} {s.get('test_f1', float('nan')):>8.4f}"
            f" {s['dp_pre']:>8.4f} {s['dp_post']:>8.4f}"
            f" {(s.get('equalized_odds_diff') or float('nan')):>7.4f}"
            f" {(s.get('equal_opportunity_diff') or float('nan')):>7.4f}"
            f" {tpr_by.get('Male', float('nan')):>6.3f}"
            f" {tpr_by.get('Female', float('nan')):>6.3f}"
            f" {s['elapsed_sec']:>6.0f}s"
        )
    print(sep)

    # Save JSON
    os.makedirs(config.RESULTS_DIR, exist_ok=True)
    batch_json = os.path.join(config.RESULTS_DIR,
                              f'{config.DATASET_NAME}_{config.BALANCE_STRATEGY}_batch_summary.json')
    with open(batch_json, 'w') as f:
        json.dump(all_summaries, f, indent=2)
    print(f'\n✓ JSON saved → {batch_json}')

    # Save CSV
    csv_rows = []
    for s in all_summaries:
        if 'error' in s:
            csv_rows.append({
                'dataset': config.DATASET_NAME,
                'seed': s.get('seed'), 'source': s.get('source'),
                'imputation': s.get('imputation', ''),
                'test_acc': None, 'test_f1': None, 'test_auc': None,
                'pre_male': None, 'pre_female': None, 'dp_pre': None,
                'post_male': None, 'post_female': None, 'dp_post': None,
                'elapsed_sec': None, 'error': s['error'],
            })
        else:
            pre  = s.get('dp_pre_group_rates',  {})
            post = s.get('dp_post_group_rates', {})
            tpr  = s.get('tpr_by_group', {}) or {}
            fpr  = s.get('fpr_by_group', {}) or {}
            csv_rows.append({
                'dataset'               : config.DATASET_NAME,
                'no_layers'             : s.get('no_layers'),
                'seed'                  : s['seed'],
                'source'                : s['source'],
                'imputation'            : s['imputation'],
                'test_acc'              : s['test_acc'],
                'test_f1'               : s.get('test_f1'),
                'test_auc'              : s.get('test_auc'),
                'pre_male'              : pre.get('Male'),
                'pre_female'            : pre.get('Female'),
                'dp_pre'                : s['dp_pre'],
                'post_male'             : post.get('Male'),
                'post_female'           : post.get('Female'),
                'dp_post'               : s['dp_post'],
                'tpr_parity_gap'        : s.get('tpr_parity_gap'),
                'fpr_parity_gap'        : s.get('fpr_parity_gap'),
                'equal_opportunity_diff': s.get('equal_opportunity_diff'),
                'equalized_odds_diff'   : s.get('equalized_odds_diff'),
                'elapsed_sec'           : s['elapsed_sec'],
                'error'                 : None,
            })

    batch_csv = os.path.join(config.RESULTS_DIR,
                             f'{config.DATASET_NAME}_{config.BALANCE_STRATEGY}_batch_summary.csv')
    pd.DataFrame(csv_rows).to_csv(batch_csv, index=False)
    print(f'✓ CSV  saved → {batch_csv}')

